In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

from langchain.agents import create_agent
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
print("\n Step 1: Creating knowledge base...")
documents = [
    Document(
        page_content="LangChain is a framework for developing applications powered by language models. It provides tools for document loading, splitting, embeddings, chains, and memory management.",
        metadata={"source": "doc1", "topic": "langchain"},
    ),
    Document(
        page_content="RAG stands for Retrieval-Augmented Generation. It combines retrieval of relevant documents with text generation. The system first searches for relevant information, then uses that context to generate informed responses.",
        metadata={"source": "doc2", "topic": "rag"},
    ),
    Document(
        page_content="Vector databases store embeddings and allow for semantic similarity search. They convert text into numerical vectors that capture meaning. Popular vector databases include ChromaDB, Pinecone, and Weaviate.",
        metadata={"source": "doc3", "topic": "vectordb"},
    ),
    Document(
        page_content="ConversationBufferMemory in LangChain stores the entire conversation history. This allows the system to maintain context across multiple turns and reference previous messages.",
        metadata={"source": "doc4", "topic": "memory"},
    ),
    Document(
        page_content="Conversational RAG combines retrieval, generation, and memory. Unlike basic RAG, it can handle follow-up questions, maintain context, and build on previous responses.",
        metadata={"source": "doc5", "topic": "conversational-rag"},
    ),
    Document(
        page_content="Google Gemini is a powerful multimodal AI model that excels at text generation, reasoning, and following instructions. Gemini Flash 2.0 is the latest and fastest version.",
        metadata={"source": "doc6", "topic": "gemini"},
    ),
    Document(
        page_content="Memory types in LangChain include ConversationBufferMemory (stores everything), ConversationSummaryMemory (stores summaries), and ConversationBufferWindowMemory (stores last K messages).",
        metadata={"source": "doc7", "topic": "memory"},
    ),
    Document(
        page_content="ChromaDB is an open-source embedding database that works well with LangChain. It's lightweight, fast, and perfect for development and prototyping.",
        metadata={"source": "doc8", "topic": "vectordb"},
    ),
]


 Step 1: Creating knowledge base...


In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = text_splitter.split_documents(documents)
print(f" Created {len(splits)} document chunks")

 Created 8 document chunks


In [4]:
print("\n Step 2: Loading embedding model...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print(" Embedding model loaded")


 Step 2: Loading embedding model...


W1027 11:32:27.859000 2555 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

 Embedding model loaded


In [5]:
print("\n Step 3: Creating vector database...")
vectorstore = Chroma.from_documents(
    documents=splits, embedding=embeddings, collection_name="conversational_rag_demo"
)
print(" Vector database created")


 Step 3: Creating vector database...
 Vector database created


In [6]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


@tool(response_format="content_and_artifact")
def search_docs(query: str):
    """Search the knowledge base for passages relevant to the query."""
    docs = retriever.invoke(query)
    content = "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in docs)
    return content, docs  # content goes to the LLM, docs go to the artifact

In [ ]:
llm = ChatOpenAI(
    model=os.getenv("GPT_MODEL", "your-gpt-model"),
    base_url=os.getenv("PORTKEY_BASE_URL", "your-base-url"),
    api_key=os.getenv("PORTKEY_API_KEY", "your-api-key"),
)

Hi there, how are you?


In [8]:
SYSTEM_PROMPT = """You are a helpful AI assistant with access to a knowledge base via `search_docs`.
- Use `search_docs` for questions about LangChain, RAG, vector DBs, memory or Gemini.
  For follow-ups, resolve references and search with a standalone query.
- Answer only from retrieved passages; otherwise say you don't know.
- Skip retrieval for greetings or questions about the conversation itself.
- Cite sources, e.g. (doc2)."""

agent = create_agent(
    model=llm,
    tools=[search_docs],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
)
config = {"configurable": {"thread_id": "chat-1"}}

In [9]:
def ask(query: str):
    return agent.invoke({"messages": [{"role": "user", "content": query}]}, config)


def chat(question, show_sources=True):
    print(f"\n You: {question}")
    msgs = agent.invoke({"messages": [HumanMessage(question)]}, config)["messages"]
    answer = msgs[-1].content
    print(f"\n Assistant: {answer}")

    last_human = max(i for i, m in enumerate(msgs) if m.type == "human")
    docs = [
        d
        for m in msgs[last_human:]
        if isinstance(m, ToolMessage) and m.artifact
        for d in m.artifact
    ]

    if show_sources:
        print(
            f"\n Sources used: {[d.metadata['source'] for d in docs] or 'none (no retrieval)'}"
        )
    print("\n" + "-" * 60)
    return {"answer": answer, "source_documents": docs}

In [15]:
result = ask("What is RAG?")
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

What is RAG?
================================== Ai Message ==================================
Tool Calls:
  search_docs (call_W76tZsrw6tC7lzd10bY7psAx)
 Call ID: call_W76tZsrw6tC7lzd10bY7psAx
  Args:
    query: RAG Retrieval-Augmented Generation
================================= Tool Message =================================
Name: search_docs

[doc2] RAG stands for Retrieval-Augmented Generation. It combines retrieval of relevant documents with text generation. The system first searches for relevant information, then uses that context to generate informed responses.

[doc5] Conversational RAG combines retrieval, generation, and memory. Unlike basic RAG, it can handle follow-up questions, maintain context, and build on previous responses.

[doc4] ConversationBufferMemory in LangChain stores the entire conversation history. This allows the system to maintain context across multiple turns and reference previ

In [12]:
# Step 14: Run a demo conversation
print("\n Running Demo Conversation...")
print("\nThis demo shows how the system maintains context across multiple turns:\n")

# Turn 1: Initial question
chat("What is RAG?")

# Turn 2: Follow-up question (uses context from Turn 1)
chat("How does it work exactly?")

# Turn 3: Another follow-up (builds on previous context)
chat("What's the difference between basic RAG and conversational RAG?")

# Turn 4: Reference to earlier conversation
chat("You mentioned memory earlier. Tell me more about the different types.")

# Turn 5: Complex follow-up
chat("Which memory type would you recommend for my use case?")


 Running Demo Conversation...

This demo shows how the system maintains context across multiple turns:


 You: What is RAG?

 Assistant: - RAG stands for Retrieval-Augmented Generation. It combines retrieval of relevant documents with text generation: first it searches for relevant information, then it uses that context to generate informed responses. (doc2)

- A variant called Conversational RAG combines retrieval, generation, and memory, enabling it to handle follow-up questions, maintain context, and build on previous responses. (doc5)

 Sources used: ['doc2', 'doc5', 'doc8']

------------------------------------------------------------

 You: How does it work exactly?

 Assistant: - RAG stands for Retrieval-Augmented Generation. It combines retrieval of relevant documents with text generation: first it searches for relevant information, then it uses that context to generate informed responses. (doc2)

- In Conversational RAG, retrieval, generation, and memory are combined, enablin

{'answer': 'Great question. Here’s a quick guide to the main LangChain memory types and when to use them, especially with Conversational RAG:\n\n- ConversationBufferMemory: stores the entire conversation history. Pros: easy to reference exact past messages and maintain full context. Cons: memory grows without bound as the chat length increases. Use when you need complete verbatim history and memory isn’t a constraint. (doc7, doc4)\n\n- ConversationSummaryMemory: keeps a summarized version of the conversation rather than the full history. Pros: much smaller memory footprint, good for very long chats. Cons: you work with summaries, so some details or exact messages may be lost. Use when long conversations are expected and you don’t need every verbatim detail. (doc7)\n\n- ConversationBufferWindowMemory: keeps only the last N messages (a sliding window). Pros: predictable memory usage and keeps recent context fresh. Cons: earlier parts of the conversation can be forgotten if not within the